### 1. Basic Tasks

In [0]:
-- 1. Connect Power BI Desktop to a Databricks SQL warehouse via Partner Connect (or a manual connection) and build one report against a gold table.

![image_1789381066582.png](./image_1789381066582.png "image_1789381066582.png")

In [0]:
-- 2. Create a Unity Catalog connection to an external PostgreSQL (or MySQL) database and a foreign catalog exposing one of its tables.

In [0]:
CREATE CONNECTION neon_postgres
TYPE POSTGRESQL
OPTIONS (
    host 'host',
    port '5432',
    user 'neondb_owner',
    password 'password'
);

In [0]:
CREATE FOREIGN CATALOG neon_catalog
USING CONNECTION neon_postgres
OPTIONS (
    database 'neondb'
);

In [0]:
select * from neon_catalog.public.admissions

In [0]:
-- 3. Read about Lakebase and write a short summary of when you'd reach for it instead of a Delta table.

**Lakebase** is useful when an application needs a **fast, operational database** for frequent small reads and writes, such as user profiles, application state, or transactions.

A **Delta table** is better for **analytics, large-scale data processing, and reporting**.

**In short:**

* **Lakebase →** application/operational workloads with fast reads and writes.
* **Delta table →** analytics, ETL, and large-scale data processing.

I would choose **Lakebase** when the data needs to be updated frequently by an application and accessed with low latency.


### 2. Intermediate Tasks

In [0]:
-- 4. Publish your Power BI report to the Power BI service and document the two ways data could go stale for this connection type (scheduled refresh vs. live query).

Data in Power BI can become stale in two main ways:

* Scheduled Refresh: Power BI imports a copy of the data. If the scheduled refresh does not run or fails, the report continues showing the previous data until the next successful refresh.
* Live Query: Power BI queries the source data when the report is accessed. Data can become stale if the source itself is not updated or if there are connection/performance issues preventing the latest data from being retrieved.

In [0]:
-- 5. Write a federated query that joins a native Unity Catalog Delta table with a foreign-catalog table from the external Postgres database, and confirm no data was physically copied first.

In [0]:
show tables in neon_catalog.public

In [0]:
SELECT
    d.name AS delta_name,
    n.first_name AS postgres_name
FROM dev.bronze.customers d
CROSS JOIN neon_catalog.public.patients n
limit 5;

6. Set up a Databricks-to-Databricks OpenShare of one gold table with a partner workspace (or
simulate the recipient side) and confirm what content types (tables, views, volumes) are supported.

In [0]:
create share if not exists sample_share;
alter share sample_share add table dev.demo.alpha;

7. (Data Analyst) Import an existing Power BI file into an AI/BI dashboard and note what did and didn't
translate cleanly.

![image_1789461930161.png](./image_1789461930161.png "image_1789461930161.png")

### 3. Advanced Tasks

8. Design a data-sharing decision matrix for Cyntexa: for a given partner scenario (has their own Databricks workspace vs. doesn't; needs tables only vs. needs AI assets), determine which OpenSharing protocol — Databricks-to-Databricks vs. Databricks-to-Open — applies and why.

1. Databricks-to-Databricks

Use this when the partner has their own Databricks workspace, for example:

* Sharing Delta tables or views with another Databricks organization.
* Sharing AI/ML assets with a partner who also uses Databricks.
* When both organizations want direct integration between Unity Catalog environments.

2. Databricks-to-Open

Use this when the partner does not have a Databricks workspace, for example:

* Sharing tables/data with a partner using PostgreSQL, Python, Spark, or another platform.
* Sharing data with external customers or vendors who do not use Databricks.
* When the partner needs to consume shared data using open Delta Sharing protocols.


9. Evaluate query federation vs. building a nightly copy pipeline for the Postgres source: under what data-freshness and query-volume conditions does federation stop making sense?

Query Federation vs. Nightly Copy Pipeline

* Use Query Federation when data needs to be fresh/near real-time and query volume is low to moderate. It avoids maintaining a separate copy of the PostgreSQL data.

* Federation stops making sense when there are many frequent queries or heavy analytical workloads against PostgreSQL, because repeated queries can put load on the source database and may have higher/less predictable latency.

* Use a Nightly Copy Pipeline when daily/stale-by-hours data is acceptable and there is high query volume or heavy analytics. The data is copied into Delta tables once per night, allowing Databricks to handle many queries efficiently without repeatedly hitting PostgreSQL.

10. Propose an LTAP architecture for a new Cyntexa feature (e.g., a real-time inventory-check app) that needs both OLTP writes (Lakebase) and OLAP analytics (Lakehouse) on the same data, specifying what syncs where and who owns each side operationally.

LTAP Architecture – Real-Time Inventory App

* Lakebase (OLTP): The inventory application writes stock updates directly to Lakebase. Lakebase is responsible for fast transactional reads/writes and current inventory state. The application/operations team owns Lakebase.

* Sync to Lakehouse (OLAP): Changes from Lakebase are continuously synced/replicated to the Lakehouse/Delta tables for analytics and reporting. The data engineering team owns the Lakehouse pipelines and ensures the analytical data stays updated.

* Lakehouse Analytics: BI dashboards and analytical queries use the Lakehouse copy rather than directly querying Lakebase. This prevents analytical workloads from affecting the application's transactional performance.

Flow:
Inventory App → Lakebase (OLTP) → Continuous Sync → Lakehouse/Delta (OLAP) → BI & Analytics

Ownership: Application/Operations → Lakebase | Data Engineering → Lakehouse